# Data representation overview

In this lecture, "data representation" is being used as an alternative to a more common term: "data structures." Where as data structures may refer to any data structure, including avl-trees, b-trees, binary-trees, in other words, low level structures defined with low level programming constructs, this lecture is only concerned with high level data structures used to represent real-world data.

Specifically, we are concernd with data represented as vectors, matrices, dataframes, sql tables - "structured" data. In modern computer science, unstructured data, such as audio, video, text is becoming very common and is now a solved problem. For the purpose of this lecture, we are more interested in the more common data format, sometimes called "tabular" data.

## 1 dimensional vectors

One dimensional data is a foundational data structure, although pehraps too low level for our use. Some examples are common lists, tuples numpy vectors and Pandas series.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
ages_list     = [34, 32, 10, 8, 2]
common_tupple = (1,2,3,4,5)

ages_array = np.array([34, 32, 10, 8, 2])
ages_s     = pd.Series([34, 32, 10, 8, 2])

More realistically, we load 2-dimensional data and extract 1 dimensional slices:

In [ ]:
ohlc_df = pd.read_csv('../../datasets/market_data/ohlcv_2025-sept.csv.zip')
ohlc_df

Extract MSFT's closing prices for the month of September (Pandas Series):

In [ ]:
msft_sept_s = ohlc_df.loc[ohlc_df.ticker == 'MSFT', 'close'].reset_index(drop=True) # We will look at indexes in a future lecture
msft_sept_s

Extract numpy vector from the Pandas series:

In [ ]:
msft_sept_v = msft_sept_s.to_numpy()
msft_sept_v

Just for good measure, let's extract a normal Python list:

In [ ]:
msft_sept_l = msft_sept_s.tolist()
msft_sept_l

### What can we do with 1-dimensional data?

#### Find prices at a specific index:

In [ ]:
msft_sept_l[0] # Get the first value

In [ ]:
msft_sept_v[0]

In [ ]:
msft_sept_s[0]

#### Get a range of values:

In [ ]:
# Get the first 3 values
msft_sept_l[0:3]

In [ ]:
msft_sept_v[0:3]

In [ ]:
msft_sept_s[0:3]

#### Aggregate over the values

In [ ]:
# Find the average closing price

import statistics

statistics.mean(msft_sept_l)

In [ ]:
msft_sept_v.mean()

In [ ]:
msft_sept_s.mean()

#### So why do we have three different ways of doing the same thing?

The most obvious answer is speed!

In [ ]:
%timeit statistics.mean(msft_sept_l) # list
%timeit msft_sept_v.mean()           # numpy
%timeit msft_sept_s.mean()           # pandas

Btw, you may have thought that numpy and pandas should be about the same speed. However, becaues pandas allows missing values, it is often slower than raw numpy.

But! Speed is not the only difference. Other than handlign missing values, pandas provides additonal functionality for hadling time series data (to be discussed later) and for providing a better developer experience.

For example, a pandas series can be given a name:

In [ ]:
msft_sept_s.name = 'MSFT September Closing Price'
msft_sept_s

And we can attach an index which gives more information than just the index location:

In [ ]:
ohlc_df2 = ohlc_df.copy() # Create a copy to keep the original unchanges
ohlc_df2 = ohlc_df2.set_index('date') # Set the date column as the index (so each column will be associated with this index)
msft_sept_s2 = ohlc_df2.loc[ohlc_df2.ticker == 'MSFT', 'close'] # Same as before, just extract MSFT's close prices

msft_sept_s2

Notice that you now have access to date based ranges, not just index locations!

In [ ]:
msft_sept_s2['2025-09-01':'2025-09-10'] # Get the first 10 days of September


## 2-dimensional matrices, dataframees and sql tables (and spreadsheets?)

Two dimensional data is several extremely powerful data structures - several of them!

### Matrices
The most obvious is to extend the mathematical vector to a matrix. We have have a name for it, a "matrix!" For our purpose, a matrix is a two dimensional array of numbers. Unlike Python's built-in lists, there is no built-in data structure for tabular data. 

Let's extract a matrix from our prices, say open and closing prices for MSFT for all of September:

In [ ]:
msft_oc_m = ohlc_df.loc[ohlc_df.ticker == 'MSFT', ['open', 'close']].to_numpy()
msft_oc_m

Like before, we can extract values from it. Let's get the opening price for the first day of September:

In [ ]:
msft_oc_m[0, 0] # Get the first row, first column (open price for the first day)

Get the closing price for the last 3 days of September:

In [ ]:
msft_oc_m[-3:, 1] # Get the last 3 rows, second column (close price for the last 3 days)

#### Something more interesting average price and average price per category
Have aggregation broken by a category gives us the opportunity to see trends and to compare values for actionable insight.

In [ ]:
msft_oc_m.mean()

Not very useful. What we if we want average open prices and average closing prices?

In [ ]:
msft_oc_m.mean(axis=0) # Get the mean across rows (for each column)

In [ ]:
msft_oc_m.mean(axis=1) # Get the mean across columns (for each row)

### Dataframes add a layer of usability on top of matrices

Notice nice column names:

In [ ]:
msft_oc_df = pd.DataFrame(msft_oc_m, columns=['open', 'close'])
msft_oc_df.head()

same as 

In [ ]:
ohlc_df.loc[ohlc_df.ticker == 'MSFT', ['open', 'close']].head()

Let's add the date index back in:

In [ ]:
# Notice that we can't just set the index to 'date', that will bring in dates even where the ticker is not MSFT
msft_oc_df = ohlc_df.loc[ohlc_df.ticker == 'MSFT', ['open', 'close']].set_index(ohlc_df.loc[ohlc_df.ticker == 'MSFT', 'date'])
msft_oc_df.head()

#### Notice the aggregate function now (it is more oppinionated and easier to use):

In [ ]:
msft_oc_df.mean()

We didn't have to tell it to aggregate per open and close column! It understood automatically that it makes no sense to find a mean of all numbers - it has a "business sense!"

In [ ]:
msft_oc_df.mean(axis='columns') # Get the mean across columns (for each row)

#### The aggregate function is even more powerful in dataframes - because of the `groupby` clause

In [ ]:
msft_oc_df['day_name'] = pd.to_datetime(msft_oc_df.index).day_name()
msft_oc_df.head(10)

In [ ]:
# Find the average open and close price for each day of the week:
msft_oc_df.groupby('day_name').mean()

#### Dataframes provide other useful primitives - let's combine this dataset with another!

In [ ]:
sectors_df = pd.read_csv('../../datasets/market_data/sectors.csv.zip', usecols=['ticker', 'industry'])
sectors_df

In [ ]:
ohlc_df.head()

In [ ]:
ohlc_sectors_df = ohlc_df.merge(sectors_df, on='ticker', how='left') # Merge the two dataframes on the ticker column (left join)
ohlc_sectors_df.head()

In [ ]:
ohlc_sectors_df.loc[ohlc_sectors_df.ticker == 'MSFT'].head()

In [ ]:
ohlc_sectors_df.loc[ohlc_sectors_df.ticker == 'F'].head()

In [ ]:
ohlc_sectors_df.loc[ohlc_sectors_df.ticker == 'YUM'].head()

#### Why don't we find the average closing price per sector?
(although, not the most sensible analysis, it is just to show the power of dataframes)

In [ ]:
ohlc_sectors_df.groupby('industry')['close'].mean()

#### Dataframe make it easy to move columns into rows and rows into columns (for example, for cross-tab analysis):

In [ ]:
ohlc_faang_df = ohlc_df[ohlc_df.ticker.isin(['META', 'AAPL', 'AMZN', 'NFLX', 'GOOG'])]
ohlc_faang_df

Show close prices per day, per ticker, in an easy to read manner

In [ ]:
ohlc_faang_df.pivot(index='date', columns='ticker', values='close')


#### Dataframes often provide built-in visualization

In [ ]:
import holoviews as hv
import hvplot.pandas  # noqa: F401  — registers the .hvplot accessor

In [ ]:
msft_oc_df.hvplot.line(y='open', label='Open Price')

#### Dataframes have other features, handling missing values, time series (to be discussed later), and more!

### SQL tables are older than dataframes and still very common. They are out of scope for this class but we do want to place them in context.

For the purpose of our class, SQL tables and relational algebra are the smae (a very controversial statement in many circles). While dataframes are loaded from raw files, sql tables live within a "database management system." They are a step or two removed from raw files. Raw files are ingested into a DBMS, which organizes data into tables, controls appends, deletion, transformations and all operations on them. Since a DBMS owns all data access, they can add optimizations that dataframes never could. For example, what we see as a two dimensional tabel of data, might actually be stored completely different - perhaps each column is a **file of its own**, perhaps the **data lives on a cluster** of 100 machines, perhaps the data is **calculated on the fly**!

#### SQL is a different language!
Here is what it looks like:

Show me all rows and all columns:
```sql
select *
from ohlc
```

Show me all rows and only the ticker, open and close columns:
```sql
select ticker, open, close
from ohlc
```

Show me all rows and only the ticker, open and close columns, **but only for MSFT**:
```sql
select ticker, open, close
from ohlc
where ticker = MSFT
```

Find the mean closing price for each ticker:
```sql
select ticker, avg(close) as avg_close
from ohlc
group by ticker
```

Join the ohlc table with the sectors table to get the industry for each ticker:
```sql
select ticker, industry
from ohlc
join sectors on ohlc.ticker = sectors.ticker
```

Find the mean closing price for each industry:
```sql
select industry, avg(close) as avg_close
from ohlc
join sectors on ohlc.ticker = sectors.ticker
group by industry
```

Show the first 5 rows of this table: ... sqls is not great for this - for one thing, rows are not supposed to have order (similar to mathematical sets)

Show the 2nd, 4th and 7th columns: ... again, sql is not great for this.

Pivot the table so the close column ... actualy many (perhaps most?) sql systems don't provide a robust implementation of pivoting

#### Did you notice that the groupby and merge features of dataframes look very similar to sql? That's where they come from!

Remember: dataframes are generally single files. You load them in your laptop, hence they are limited by the resources on your computer. There are many alternatives and derivatives which let you improve the speed, run them on a gpu, run them across a cluster, etc. But generally speaking, dataframes are extremely handy and a swiss army knife of analysis but are limited by how much data they can process.

SQL tables, on the other hand, have been designed to handle extremely large amounds of data. Databases are often run by dedicated teams.

General advice: **You shold prefer to do as much data processing via sql as possibe, before switching to dataframes**

## N-dimensional data (tensors)
You may have thought that we left numpy behind. Recall that it was faster, but aren't dataframes better for everything else? Numpy (and related libraries, such as Pytorch, MLX, Jax) are designed to handle n-dimensional data.

Imagine a picture: it has a height and width, but it also has a color channel, remmeber RGB?. A typical picture is actually a 3d matrix! See the numpy lecture for a nice example.

We can demonstrate this with our financial data as well. Let's convert our ohlc dataframe to a 3d matrix where rows represent tickers, columns represend open, highl, low, close prices and the z-axis represends dates:

In [ ]:
ohlc_df.head()

In [ ]:
ohlc_df.ticker.nunique() # Get the unique tickers

In [ ]:
ohlc_df.date.nunique() # Get the unique dates

Let's build our 3d numpy matrix
(note that often you will not have to build such matrices yourself. However, you will need to understand them and how to use them)

In [ ]:
matrix_per_date = list()

ROWS    = ['AAPL', 'AMZN', 'NFLX', 'GOOG']
COLUMNS = ['ticker', 'date', 'open', 'close']

for key, group in ohlc_df.loc[ohlc_df.ticker.isin(ROWS), COLUMNS].groupby('date'):
    #print(key)
    # Extract matrix where each row represents AAPL, AMZN, GOOD, NFLX and each column represents OPEN, CLOSE prices
    matrix_of_open_close = group.sort_values(by='ticker')[['open', 'close']].to_numpy()
    matrix_per_date.append(matrix_of_open_close)

In [ ]:
threed_mat = np.stack(matrix_per_date, axis=2)#.shape
threed_mat.shape

As the `.shape` attribute above shows, `threed_mat` is a 3 dimensional matrix. 
- Dimension zero has 4 entries (rows represent the 4 tickers)
- Dimension one has 2 entries (columns represent open and close prices)
- Dimension two has 21 entries (depth represents the 21 trading days of September)


In [ ]:
# All tickers, open/close values for the first day of September
threed_mat[:, :, 0]

Let's double check the numbers:

In [ ]:
ohlc_df.loc[(ohlc_df.date == "2025-09-02") & (ohlc_df.ticker.isin(ROWS)), COLUMNS]

Checks out!

This is what the 3d matrix looks like in raw form:

In [ ]:
threed_mat

Yikes! Better to think in terms of its shape:

In [ ]:
threed_mat.shape

Let's find the average open and close price for all symbols, across all days:

In [ ]:
threed_mat.mean(axis=2) 

Unless you work with tensors daily, even something as simple as finding the right axis to aggregate over can be confusing. It is important to get in the habit of doing sanity checks. **The first check is, does the `.shape` make sense?** If we want to find the mean open and mean close price for each symbol, the resulting table should still have 4 rows (representing each symbol and 2 columns (representing open and close price):

In [ ]:
threed_mat.mean(axis=2).shape

Looks reasonable!

Let's spot check the numbers:

In [ ]:
ohlc_df.loc[(ohlc_df.ticker == 'AAPL'), COLUMNS].agg({'open': 'mean', 'close': 'mean'})

In [ ]:
ohlc_df.loc[(ohlc_df.ticker == 'NFLX'), COLUMNS].agg({'open': 'mean', 'close': 'mean'})

The first and last rows do indeed match, we should feel much more confident about our calculation.

#### Note that we just calculated means for the whole month, across many tickers, for two different price categories, _in a single, short, command!_
Compare this to the spot check we did with Pandas. Numpy is much more succint (and faster). For experts, it brings immense power! However, for non-experts, this approach can be a bit too concise. 

## Time dimension enables additonal functionality

#### Even basic Python lists can be thought of as supporting "time" or order based features:

Get the **last three** values:

In [ ]:
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10][-3:]

Get the **first three characters** of a string:

In [ ]:
"Mr. Homer Simpson"[:3]

#### Basic numpy vectors have some support for time series data:
Find the cumulative sum of a vector at each point:

In [ ]:
np.arange(10).cumsum()

More interestingly, given the matrix of open/close prices across several days for AAPL, we can find the change in price as such:

In [ ]:
dummy_prices = np.array([
    # Open           Close
    [241, 242],  # Day 1
    [229, 228],  # Day 2
    [242, 242 ], # Day 3
    [122, 122]   # Day 4
])

dummy_prices

In [ ]:
# Find changes in prices from one day to the next:
dummy_prices[1:, :] - dummy_prices[:-1, :]


All of a sudden, you are on your way to making prices for arbitrary number of symbols stationary and ready for further analysis. 

(note that proper stationarization is a bit more involved. For one thing, you are likely to do a diff over logs, but this is a good start)


### Pandas series and dataframes have a rich set of time series features.


Recall from earlier that we set the date column as the index:

In [ ]:
ohlc_df2 = ohlc_df.copy() # Create a copy to keep the original unchanges
ohlc_df2.date = pd.to_datetime(ohlc_df2.date) # Convert the date column to datetime format
ohlc_df2 = ohlc_df2.set_index('date') # Set the date column as the index (so each column will be associated with this index)
ohlc_df2 = ohlc_df2.sort_index() # Sort dataset by index (unfortunately, necessary)

In [ ]:
ohlc_df2

#### We can use the sample method to resample the data to a different frequency. 
Instead of daily data, say we want it every 3 days, and we instruct Pandas to do so by taking a mean of daily values

In [ ]:
apple_close_px_df = ohlc_df2.loc[ohlc_df2.ticker == 'AAPL', ['close']]
apple_close_px_df.head()

In [ ]:
apple_close_px_df.resample('3d').mean()

#### We can use the `shift` operator to find the change in price from one day to the next:

In [ ]:
# shift(1) brings yesterday's closing price down to today's row
apple_close_px_df - apple_close_px_df.shift(1)

#### Almost just as easily, we can find moving averages with the `rolling` operator:

In [ ]:
# Takes the last 20 rows, calculates the mean, and puts it on the current row
apple_close_px_df['close'].rolling(window=3).mean()

In [ ]:
# You only need to import this once at the top of your notebook
import hvplot.pandas 

# 1. Calculate the moving average (same as before)
apple_close_px_df['SMA_3'] = apple_close_px_df['close'].rolling(window=3).mean()

# 2. The hvplot magic! 
# By passing a list to 'y', it plots both columns on the same chart automatically.
apple_close_px_df.hvplot.line()

In [ ]:
apple_close_px_df.head()